In [1]:
import datasets
import tensorflow as tf
from transformers import AutoTokenizer
from datasets import load_dataset
datasets.config.AUDIO_DECODE_BACKEND = "torchaudio"
ds = load_dataset("neerajaabhyankar/hindustani-raag-small", streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/327 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1161 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1161 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

In [2]:
print(ds)

IterableDatasetDict({
    train: IterableDataset({
        features: ['audio', 'label'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['audio', 'label'],
        num_shards: 1
    })
})


In [3]:
print(ds['train'].features)

{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'label': ClassLabel(names=['AheerBhairav', 'AlhaiyaBilawal', 'Bageshree', 'Bahar', 'Bairagi', 'Basant', 'Bhairav', 'Bhairavi', 'Bheempalasi', 'Bhoopali', 'Bihag', 'Chandrakauns', 'Charukeshi', 'DarbariKanada', 'Des', 'Deshkar', 'Dhani', 'Durga', 'Hameer', 'HansDhwani', 'Hindol', 'Jaijaivanti', 'Jog', 'Kafi', 'Kalawati', 'KaushikDhwani', 'Kedar', 'Keerwani', 'Khamaj', 'Lalit', 'Madhukauns', 'Madhuvanti', 'Malhar', 'Malkauns', 'MaruBihag', 'Marwa', 'Multani', 'Pilu', 'PuriyaDhanashri', 'PuriyaKalyan', 'Sarang', 'Shankara', 'Shivranjani', 'Shree', 'Sohani', 'TilakKamod', 'Tilang', 'Todi', 'Vibhas', 'Yaman'])}


In [4]:
example = ds['train'][0]
print(example)

In [5]:
print(ds['train']['sentence'][:5])

In [6]:
# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_function(examples):
    return tokenizer(examples['sentence'], truncation=True, padding=True)
tokenized_dataset = ds.map(tokenize_function, batched=True)

# Remove original columns if they are no longer needed
tokenized_dataset = tokenized_dataset.remove_columns(["sentence"])

print(tokenized_dataset)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

IterableDatasetDict({
    train: IterableDataset({
        features: Unknown,
        num_shards: 1
    })
    test: IterableDataset({
        features: Unknown,
        num_shards: 1
    })
})


In [7]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

In [8]:
tokenized_dataset.with_format("tf")

IterableDatasetDict({
    train: IterableDataset({
        features: Unknown,
        num_shards: 1
    })
    test: IterableDataset({
        features: Unknown,
        num_shards: 1
    })
})

In [9]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [10]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

data_loader = DataLoader(tokenized_dataset, collate_fn=collate_fn, batch_size=4)

In [11]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [12]:
import librosa
import numpy as np
import torch

# Define parameters for the spectrogram
SAMPLE_RATE = 16000 # All audio should be resampled to this
N_FFT = 1024        # Number of samples for FFT
HOP_LENGTH = 512    # Sliding window hop size
N_MELS = 128        # Number of Mel bands
MAX_LEN_SECONDS = 1 # Max duration of audio clips in seconds
MAX_LEN_SAMPLES = int(MAX_LEN_SECONDS * SAMPLE_RATE)

def preprocess_function(examples):
    # The 'audio' field contains a batch of audio file paths or arrays
    audio_arrays = [x["array"] for x in examples["audio"]]

    # Pad or truncate audio to a fixed length
    def pad_or_truncate(array, length):
        if len(array) < length:
            padding = length - len(array)
            return np.pad(array, (0, padding), mode='constant')
        else:
            return array[:length]

    # Apply padding/truncation
    padded_arrays = [pad_or_truncate(arr, MAX_LEN_SAMPLES) for arr in audio_arrays]

    # Convert audio arrays to Mel spectrograms
    # The output shape will be (batch_size, n_mels, time_steps)
    spectrograms = [
        librosa.feature.melspectrogram(
            y=arr,
            sr=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS
        )
        for arr in padded_arrays
    ]

    # Convert power spectrograms to decibels (dB)
    # This is a common practice that can improve model performance
    log_spectrograms = [librosa.power_to_db(s, ref=np.max) for s in spectrograms]
    processed_spectrograms = [np.expand_dims(s, axis=0) for s in log_spectrograms]

    # Convert lists of numpy arrays to PyTorch tensors
    input_values = torch.tensor(np.array(processed_spectrograms), dtype=torch.float32)
    labels = torch.tensor(examples["labels"], dtype=torch.long)

    # Return the processed spectrograms and labels
    return {"input_values": input_values, "labels": labels}

In [16]:
import datasets
from datasets import Audio

datasets.config.AUDIO_DECODE_BACKEND = "torchaudio"
# Ensure the audio is cast to the correct sampling rate first
datasets = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))